# VMD-MFGNN — N=15 Consolidated Run (Colab)

This notebook produces every remaining result for Paper 1 (VMD-MFGNN, copper
price forecasting) in one unattended, overnight, resumable Colab run. It
**supersedes** the old N=8 notebook
(`notebooks/archive_paper1_vmd_mfgnn/vmd_mfgnn_v2_colab.ipynb`) and the prior
content of this same file (an earlier "robustness experiments" notebook,
fully overwritten here — none of its content is reused, only its Drive-sync
helper *patterns*, described below).

**What changed this session, all already committed to `src/` (this notebook
only orchestrates it, it does not implement any of it):**
- Dataset expanded from N=8 to N=15 variables (aluminum removed — its real
  Yahoo Finance history starts 2014-05-06, not 2010, and was silently
  truncating the whole panel; 4 new yfinance tickers + 4 new FRED series
  added).
- GATv2 swap in the per-band graph attention layer.
- A temperature-parameterized graph-fix (`use_graph_temperature`), exercised
  via a dedicated ablation variant (`full_model_graphfix_temp`), OFF by
  default in the main model.
- A 20-LSTM multi-horizon architecture (one shared graph across all 4
  horizons, a separate LSTM + fusion + head per horizon).
- A faithful MTGNN baseline reproduction (`MTGNNBaseline`), alongside the
  existing lower-fidelity `SimpleMTGNN`.
- A corrected, genuinely univariate VMD-LSTM baseline.
- A critical bug fix in `BandGraphEncoder._batch_edge_index`: graph edges are
  now correctly wired at `batch_size > 1` — they never were before, which
  means every previously-reported ablation/null-control result computed with
  a batch size above 1 measured an inert graph.
- A `min_epochs` floor (30), now threaded through VMD-MFGNN, every baseline,
  and every ablation variant, preventing an early-stopping artifact where a
  variant's "best" checkpoint landed at epoch 0-5.
- A null-control experiment (real VMD input vs. shuffled vs. Gaussian-noise
  non-target channels) testing whether the per-band graph's collapse is
  downstream of VMD decomposition specifically, or would happen for any
  input.

**Two-phase design — read before running.** Data preparation (yfinance
download + FRED pull + VMD decomposition) is a CPU-only step with no GPU
path and nothing to gain from a Colab GPU runtime (VMD's ADMM optimization
is pure CPU numpy). It was run **locally on the user's own machine**, not in
this notebook. Before running this notebook, upload the resulting
`data/raw_prices.csv`, `data/vmd_modes.npy`, and `data/vmd_modes_meta.json`
to Google Drive at `MyDrive/Copper_Paper1.2/data_cache/` — the "Data
Acquisition" section below (Section 3) restores from that cache automatically
if present. **This is an optimization, not a requirement**: if the cache
isn't there, Section 3 still works correctly, it just re-downloads and
re-decomposes from scratch (slow — see that section's own timing note).

**Resumability, in one sentence.** Every training step (HPO trials,
VMD-MFGNN, each of 7 baselines, each of 8 ablation variants, each of 3
null-control arms) checkpoints to local disk and is synced to Google Drive
**immediately after it completes** — not just at the end of a section — so a
Colab disconnect at any point loses at most the one unit of work that was
in-flight when it happened, and a fresh session picks up exactly where the
old one left off with zero human intervention.

**No decisions left for whoever runs this.** Every hyperparameter comes from
the checked-in `configs/default.yaml` (N=15 tickers, `min_epochs: 30`,
`horizon_loss_weighting: "target_variance"`, `hpo.enabled: true`). There is
no manual "choose an option" cell anywhere in this notebook.


## Section 1: Setup

Mount Drive, get the repo at the correct branch, install dependencies,
confirm the GPU runtime, and sanity-import every module this notebook uses.


### 1.1 Mount Google Drive

`Copper_Paper1.2` is a deliberately NEW, non-overlapping Drive directory name — this N=15 run's results must never collide with the prior N=8 run's `Copper_Paper1` directory.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/Copper_Paper1.2'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive results root:', DRIVE_ROOT)

### 1.2 Get the repo

Pick ONE of the two cells below (both are provided; uncomment the one you
want and leave the other commented). Either way you end up with the repo at
`/content/copper`, on branch `graph-fix-experiment` (confirmed as the actual
current branch via `git branch --show-current` at the time this notebook was
built — re-confirm locally with the same command before relying on a fresh
clone here, in case the branch has since changed or been merged to `main`).

In [ ]:
# ---- OPTION A: git clone/pull from the real remote (recommended) ----
REPO_URL = 'https://github.com/anmol0705/Copper_Price_Forecasting.git'
BRANCH = 'graph-fix-experiment'  # confirm this is still current: `git branch --show-current` locally

import os, subprocess
if not os.path.exists('/content/copper'):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, '/content/copper'], check=True)
else:
    subprocess.run(['git', '-C', '/content/copper', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/copper', 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/copper', 'reset', '--hard', f'origin/{BRANCH}'], check=True)
print('Repo ready at /content/copper on branch', BRANCH)

In [ ]:
# ---- OPTION B: upload a copper.zip (use if you haven't pushed the branch yet) ----
# 1. Zip the repo locally: `cd D:\copper && zip -r copper.zip . -x '.git/*'`
# 2. Run this cell, click "Choose Files", select copper.zip.
# 3. Uncomment the extraction lines.

# from google.colab import files
# uploaded = files.upload()  # select copper.zip
# import zipfile
# with zipfile.ZipFile('copper.zip', 'r') as zf:
#     zf.extractall('/content/copper')
# print('Extracted to /content/copper')

In [ ]:
import os, sys
assert os.path.isdir('/content/copper'), (
    "Repo not found at /content/copper -- run Option A or Option B above first."
)
os.chdir('/content/copper/paper1')  # all paths below (data/, results/, configs/) are relative to paper1/
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())
print(sorted(os.listdir('.')))

### 1.3 Install dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q vmdpy optuna yfinance torch-geometric EMD-signal

### 1.4 Confirm GPU runtime, sanity-import everything this notebook uses

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected -- Colab Runtime > Change runtime type > GPU. '
          'CPU-only will still run correctly (nothing below hard-requires a GPU), '
          'just much slower for VMD-MFGNN/baseline/ablation training.')

# Sanity-import every module this notebook drives, so an import-time failure
# (a typo, a missing dependency, a stale src/ tree) surfaces HERE, in the
# first minute, rather than hours into an unattended overnight run.
from src.utils import load_config, set_seed, get_device, compute_metrics, save_results
from src.data_pipeline import create_datasets
from src.models.vmd_mfgnn import VMDMFGNN
from src.models.pooled_graph_mfgnn import PooledGraphMFGNN, count_parameters
from src.models.baselines import (
    ARIMABaseline, LSTMBaseline, TransformerBaseline, SimpleMTGNN,
    MTGNNBaseline, XGBoostBaseline, VMDLSTMBaseline,
)
from src.trainer import (
    VMDMFGNNTrainer, run_baseline, run_significance_tests, print_results_table,
    _UnsqueezeModeWrapper, _find_matched_hidden_dim,
)
from src import hpo as hpo_mod
from src import experiments as exp
from src import diagnostics as diag_mod
from src.visualize import generate_all_figures
print('All modules import cleanly.')

## Section 2: Drive sync helpers

Reused **verbatim** (only `DRIVE_ROOT` is adapted) from this notebook's own
prior content -- these helpers are already correct and battle-tested:
content-addressed, unconditional-overwrite sync (no size/mtime staleness
heuristic -- deliberately avoiding that documented bug class), and per-cell
(not per-section) sync cadence so a long-running grid never leaves more than
one in-flight unit of work unsynced.


In [ ]:
import shutil, time as _time

# `data/` (raw_prices.csv + the VMD modes cache) is deliberately NOT under
# results/, and it is git-ignored -- so on a fresh Colab session it would
# otherwise be rebuilt from scratch every single time: a fresh yfinance/FRED
# download plus real VMD decomposition (CPU-only, no GPU path -- see the
# title cell's two-phase-design note). Backing it up to Drive turns that into
# a copy, and -- more importantly than the time -- it PINS the exact price
# array across sessions. build_decomposed_modes keys its cache on a sha256 of
# the price array, so if a later yfinance pull revised or backfilled even one
# row, every restored modes cache would silently invalidate AND a later
# session's models would be trained on different data than an earlier one's.
DRIVE_DATA_DIR = os.path.join(DRIVE_ROOT, 'data_cache')


def sync_to_drive(local_dir, note=''):
    """Unconditionally copies `local_dir` (any directory under `results/`,
    e.g. 'results', 'results/checkpoints', 'results/robustness/null_control')
    to the matching path under DRIVE_ROOT. Always overwrites -- deliberately
    no size/mtime freshness check (that heuristic is a documented bug class:
    a same-sized-but-different file, or a clock skew between the Colab VM and
    Drive, silently masks a genuine change)."""
    if not os.path.isdir(local_dir):
        print(f'{local_dir} does not exist yet, nothing to sync')
        return
    rel = os.path.relpath(local_dir, 'results')
    dest = os.path.join(DRIVE_ROOT, rel)
    os.makedirs(os.path.dirname(dest) or '.', exist_ok=True)
    shutil.copytree(local_dir, dest, dirs_exist_ok=True)
    print(f'[{_time.strftime("%H:%M:%S")}] synced {local_dir} -> {dest} {note}')


def restore_from_drive_if_present(local_dir):
    """Opposite direction: pulls a prior Drive-backed copy down into the
    fresh clone's local_dir, if one exists, so a resumed session picks up
    completed work from before a disconnect. Always overwrites local with the
    Drive copy -- same no-staleness-heuristic principle."""
    rel = os.path.relpath(local_dir, 'results')
    src = os.path.join(DRIVE_ROOT, rel)
    if os.path.isdir(src):
        os.makedirs(local_dir, exist_ok=True)
        shutil.copytree(src, local_dir, dirs_exist_ok=True)
        print(f'Restored {src} -> {local_dir}')
    else:
        print(f'No prior Drive backup at {src}, starting fresh')


def sync_data_cache_to_drive(note=''):
    """Backs up data/ (raw_prices.csv + vmd_modes.npy + its _meta.json
    sidecar) to Drive. NOTE: this deliberately does NOT go through
    sync_to_drive() -- that helper computes os.path.relpath(local_dir,
    'results'), which for 'data' yields '../data' and would write OUTSIDE
    DRIVE_ROOT. Separate helper, separate Drive subdirectory."""
    if not os.path.isdir('data'):
        print('no data/ yet, nothing to sync')
        return
    os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
    n = 0
    for fname in sorted(os.listdir('data')):
        p = os.path.join('data', fname)
        if os.path.isfile(p):
            shutil.copy2(p, os.path.join(DRIVE_DATA_DIR, fname))
            n += 1
    print(f'[{_time.strftime("%H:%M:%S")}] synced {n} data-cache file(s) -> '
          f'{DRIVE_DATA_DIR} {note}')


def restore_data_cache_from_drive():
    """Restores data/ from Drive on a fresh runtime. Presence-based only
    (copies a Drive file down when the local one is absent) -- these caches
    are write-once, so there is no freshness ambiguity to get wrong, and no
    size/mtime heuristic is used (that heuristic is the exact documented bug
    class this design avoids)."""
    if not os.path.isdir(DRIVE_DATA_DIR):
        print(f'No prior Drive data cache at {DRIVE_DATA_DIR}, starting fresh '
              f'(expect a fresh yfinance/FRED pull + full VMD decomposition -- '
              f'see the title cell\'s two-phase-design note; upload the local '
              f'data/ cache to this Drive path to skip that cost)')
        return
    os.makedirs('data', exist_ok=True)
    n = 0
    for fname in sorted(os.listdir(DRIVE_DATA_DIR)):
        srcf = os.path.join(DRIVE_DATA_DIR, fname)
        dstf = os.path.join('data', fname)
        if os.path.isfile(srcf) and not os.path.exists(dstf):
            shutil.copy2(srcf, dstf)
            n += 1
    print(f'Restored {n} data-cache file(s) from {DRIVE_DATA_DIR}')


def make_cell_sync(results_dir, label):
    """Builds the `on_cell_done` callback handed to src/experiments.py's
    resumable-grid drivers (e.g. run_null_control_experiment). Fires after
    EACH grid cell's jsonl row is flushed to local disk, and pushes that
    cell's results dir (jsonl + checkpoint + prediction .npy) AND any
    newly-built decomposition cache to Drive right then -- the difference
    between losing one in-flight cell and losing every completed-but-unsynced
    cell when Colab disconnects mid-grid."""
    def _on_cell_done(cell_key, result):
        sync_to_drive(results_dir, note=f'after {label} cell {tuple(cell_key)}')
        sync_data_cache_to_drive(note=f'after {label} cell {tuple(cell_key)}')
    return _on_cell_done


def jsonl_progress(jsonl_path):
    import json as _json
    if not os.path.exists(jsonl_path):
        print(f'{jsonl_path}: no results yet.')
        return {}
    seen, errors = {}, 0
    with open(jsonl_path) as f:
        for line in f:
            try:
                r = _json.loads(line)
            except Exception:
                continue
            key = tuple(r.get('cell_key', []))
            seen[key] = r
            if 'error' in r:
                errors += 1
    print(f'{jsonl_path}: {len(seen)} completed cells (errored: {errors})')
    for k in sorted(seen):
        print('  ', k)
    return seen


def load_rows_from_jsonl(jsonl_path):
    """Reads ALL rows from a *_results.jsonl file directly off disk (via
    exp.load_completed_cells, with the SAME completed-checkpoint artifact
    check every experiment uses), independent of whatever is or isn't still
    in this kernel's memory."""
    completed = exp.load_completed_cells(
        jsonl_path, artifact_check_fn=lambda row: exp._checkpoint_is_complete(
            row.get('checkpoint_path', '')))
    return list(completed.values())


# Pull back anything from a previous (possibly interrupted) session before
# doing any work below.
restore_from_drive_if_present('results')

## Section 3: Data Acquisition

Restores the data cache from Drive first (see the title cell's two-phase-
design note -- the expensive VMD decomposition was run locally, not here),
then builds the N=15 dataset via `create_datasets`. Followed by hard,
fail-loud assertions: this must never silently proceed on a wrong dataset.

**Wall-clock estimate:** ~1-3 min if the Drive cache is present (warm), or
several hours if it is not (cold -- a fresh yfinance/FRED download plus
real, leakage-safe, true-daily-refit rolling-window VMD decomposition over
~4,000 trading days x 15 variables; see `src/data_pipeline.py`'s
`VMDDecomposer` docstring). Uploading the local `data/` cache to
`MyDrive/Copper_Paper1.2/data_cache/` before running this notebook is
strongly recommended.


In [ ]:
import pandas as pd

restore_data_cache_from_drive()

config = load_config('configs/default.yaml')
set_seed(config['training']['seed'])

print('Date range:', config['data']['start_date'], '->', config['data']['end_date'])
print('Tickers:', list(config['data']['tickers'].keys()))
print('VMD K:', config['vmd']['K'])
print('training.min_epochs:', config['training']['min_epochs'])
print('training.horizon_loss_weighting:', config['training']['horizon_loss_weighting'])
print('hpo.enabled:', config['hpo']['enabled'])

data = create_datasets(config, debug_fast=False)

print('Variables:', data['variable_names'])
print(f"num_vars={data['num_vars']}, num_modes={data['num_modes']}")
print('Train/Val/Test samples:', len(data['train_ds']), len(data['val_ds']), len(data['test_ds']))

# ---- Hard, fail-loud assertions -- never continue past a wrong dataset ----

# N=15 scope check. This supersedes the old N=8 (and briefly-considered N=16)
# scope. Aluminum (ALI=F) was deliberately dropped this session: its real
# Yahoo Finance history starts 2014-05-06, not 2010, and the pipeline's
# `ffill(limit=5).dropna()` step would otherwise silently truncate the ENTIRE
# panel to 2014-2025 -- a ~4.3 year, ~25% data loss -- while main.tex's text
# and the 2010-2019/2020-2021/2022-2025 train/val/test split boundaries
# assume the panel starts 2010-01-04. See src/data_pipeline.py's
# DataDownloader.download() stale-cache ValueError and the TICKERS dict
# comment for the full history of this exact failure mode.
assert data['num_vars'] == 15, (
    f"Expected exactly 15 variables (11 yfinance tickers + 4 FRED series, "
    f"aluminum excluded), got {data['num_vars']}: {data['variable_names']}. "
    f"This almost certainly means either a ticker/FRED series silently "
    f"failed to download (see DataDownloader.download()'s raise-on-missing "
    f"logic) or a stale pre-N=15 data/raw_prices.csv cache slipped through "
    f"restore_data_cache_from_drive() without tripping its own ValueError "
    f"guard. Refusing to proceed -- fix the data before training anything on it."
)
assert 'aluminum' not in data['variable_names'], (
    "'aluminum' (ALI=F) is present in the downloaded panel -- this is the "
    "REMOVED ticker (see the TICKERS dict comment in src/data_pipeline.py: "
    "its real history starts 2014-05-06, not 2010, and previously silently "
    "truncated the whole panel via ffill(limit=5).dropna()). A stale cache "
    "or an un-updated TICKERS dict is in play; refusing to proceed."
)

x0, y0 = data['train_ds'][0]
expected_shape = (config['data']['lookback'], config['vmd']['K'], data['num_vars'])
assert tuple(x0.shape) == expected_shape, (
    f"Sample X shape {tuple(x0.shape)} != expected {expected_shape} "
    f"(lookback, K, num_vars). A mismatch here means the VMD modes cache and "
    f"the raw price panel disagree on num_vars/K (e.g. a stale vmd_modes.npy "
    f"left over from a different variable count) -- do not proceed."
)

# Date-range check -- catches any future silent-truncation bug the same way
# the aluminum one was caught this session (see DataDownloader.download()'s
# own stale-cache guard, which this assertion deliberately echoes: a later
# start date than expected means some column is again forcing dropna() to
# truncate the panel).
first_date = data['prices'].index.min()
expected_first = pd.Timestamp('2010-01-04')
assert abs((first_date - expected_first).days) <= 7, (
    f"data/raw_prices.csv's first date is {first_date.date()}, expected ~"
    f"{expected_first.date()} (the panel's actual first trading day now that "
    f"aluminum -- whose 2014-05-06 start previously forced a silent "
    f"truncation of the whole panel -- is excluded; see the TICKERS dict "
    f"comment in src/data_pipeline.py). A later start date here is exactly "
    f"the failure mode diagnosed and fixed this session: some column is "
    f"again forcing dropna() to truncate the panel. Refusing to proceed -- "
    f"identify and fix the offending column before training on this data."
)
print('All data sanity assertions passed.')

sync_data_cache_to_drive()

## Section 4: Hyperparameter Optimization (VMD-MFGNN only)

Controlled entirely by `configs/default.yaml`'s `hpo:` block (`enabled: true`,
`n_trials: 10`, `trial_epochs: 25` as checked in -- read from `config` below,
never hardcoded here). HPO is scoped to VMD-MFGNN only: the 8 baselines and
all 8 ablation variants are never tuned (see `configs/default.yaml`'s own
comment on this asymmetry).

Resumable at two layers, so a Colab disconnect mid-HPO never restarts from
zero:
1. `results/hpo_study.db` is a durable Optuna SQLite study, synced to Drive
   after **every trial** via `on_trial_done` -- a disconnect loses at most
   the one in-flight trial.
2. If `results/hpo_best_params.json` already exists (e.g. restored from
   Drive because a prior session finished HPO entirely), Optuna is skipped
   altogether and the saved winner is reused.

The `vmd_mfgnn_config`/`mc` construction below replicates
`src/trainer.py`'s `run_all_experiments()` HPO-override block exactly
(field-by-field), so this notebook's manually-driven training and the
library's own orchestration path produce equivalent behavior.


In [ ]:
import json
import copy
from pathlib import Path

hpo_results_path = Path('results/hpo_best_params.json')

if config.get('hpo', {}).get('enabled', False):
    if hpo_results_path.exists():
        print(f'[resume] Found existing {hpo_results_path} (likely restored from '
              'Drive by restore_from_drive_if_present() above) -- skipping run_hpo() '
              'and reusing the already-completed HPO winner instead.')
        with open(hpo_results_path) as f:
            best_params = json.load(f)
        print('HPO winner hyperparameters (loaded from disk):', best_params)
    else:
        n_trials = config['hpo'].get('n_trials', 15)
        trial_epochs = config['hpo'].get('trial_epochs', 25)
        print(f'HPO enabled: running Optuna search ({n_trials} trials, '
              f'{trial_epochs} epochs/trial) for VMD-MFGNN only...')

        def _hpo_trial_sync(study_, trial_):
            # Fires after EVERY trial (both a freshly-run one and one
            # restored from the resumable sqlite study) -- keeps the study
            # database and any partial trials_csv on Drive current, so a
            # disconnect never loses more than the one in-flight trial.
            sync_to_drive('results', note='after HPO trial')

        best_params = hpo_mod.run_hpo(
            config, data, n_trials=n_trials, trial_epochs=trial_epochs,
            storage_path='results/hpo_study.db',
            study_name='vmd_mfgnn_hpo',
            on_trial_done=_hpo_trial_sync,
        )
        print('HPO winner hyperparameters:', best_params)

    # Mirrors src/trainer.py's run_all_experiments() HPO-override block
    # exactly (same fields, same source dict, same order), so this
    # notebook's manually-driven training loop and the library's
    # run_all_experiments() path produce equivalent behavior when HPO is
    # enabled.
    mc = dict(config['model'])
    mc['hidden_dim'] = best_params['hidden_dim']
    mc['num_heads'] = best_params['num_heads']
    mc['dropout'] = best_params['dropout']
    mc['num_gnn_layers'] = best_params['num_gnn_layers']
    vmd_mfgnn_config = copy.deepcopy(config)
    vmd_mfgnn_config['model'] = mc
    vmd_mfgnn_config['training']['learning_rate'] = best_params['learning_rate']

    print('VMD-MFGNN will train with the HPO-tuned model/training config above.')
    print('Baselines and ablation variants below are unaffected (never HPO-tuned by design).')
    sync_to_drive('results', note='after HPO section (hpo_best_params.json / hpo_study.db)')
else:
    print("hpo.enabled=False in configs/default.yaml -- VMD-MFGNN will train with "
          "the hidden_dim/num_heads/dropout/num_gnn_layers/learning_rate values "
          "from configs/default.yaml as-is.")
    mc = config['model']
    vmd_mfgnn_config = config

## Section 5: Main Results -- VMD-MFGNN + 7 Baselines

VMD-MFGNN (checkpointed natively via `VMDMFGNNTrainer.fit(checkpoint_path=...)`)
plus 7 baselines: ARIMA, LSTM, Transformer, SimpleMTGNN, MTGNN, XGBoost (all
on raw price loaders) and VMD-LSTM (on VMD-mode loaders). Every model is
independently resumable and synced to Drive immediately after it finishes,
so a disconnect mid-section only ever loses the one model that was training
when it happened.

`base_cfg` below is copied field-for-field from `src/trainer.py`'s
`run_all_experiments()` (including `min_epochs`, threaded through every
baseline via `TorchBaseline.fit`), and deliberately built from the
**untuned** `config`, not `vmd_mfgnn_config` -- baselines are never HPO-tuned.


In [ ]:
from pathlib import Path
import time
import numpy as np
import torch

from src.utils import compute_metrics, get_device, save_results, set_seed

seed = config['training']['seed']
set_seed(seed)
device = str(get_device())
horizons = data['horizons']
num_vars = data['num_vars']
num_modes = data['num_modes']

# Copied field-for-field from src/trainer.py's run_all_experiments() base_cfg
# (lines ~577-591), built from the UNTUNED config -- baselines are never
# HPO-tuned (see configs/default.yaml's hpo: block comment).
base_cfg = {
    'num_vars': num_vars, 'num_modes': num_modes,
    'hidden_dim': config['model']['hidden_dim'],
    'num_layers': config['model']['temporal_layers'],
    'num_heads': config['model']['num_heads'],
    'dropout': config['model']['dropout'],
    'lookback': config['data']['lookback'],
    'horizons': horizons,
    'epochs': config['training']['epochs'],
    'lr': config['training']['learning_rate'],
    'weight_decay': 1e-5,
    'patience': config['training']['patience'],
    'min_epochs': config['training'].get('min_epochs', 0),
    'device': device,
}
print('base_cfg:', base_cfg)

pred_dir = Path('results/predictions')
interp_dir = Path('results/interpretability')
pred_dir.mkdir(parents=True, exist_ok=True)
interp_dir.mkdir(parents=True, exist_ok=True)

all_results = {}


def _predictions_exist(name, horizons_):
    return all(
        (pred_dir / f'{name}_{h}.npy').exists() and (pred_dir / f'{name}_{h}_true.npy').exists()
        for h in horizons_
    )


def _metrics_from_saved_predictions(name, horizons_):
    res = {'name': name}
    for h in horizons_:
        pred = np.load(pred_dir / f'{name}_{h}.npy')
        true = np.load(pred_dir / f'{name}_{h}_true.npy')
        res[f'h{h}'] = compute_metrics(true, pred)
    return res

In [ ]:
# ---- VMD-MFGNN (our model) ----
# Checkpointing is trainer-native (VMDMFGNNTrainer.fit(checkpoint_path=...)):
# if results/checkpoints/vmd_mfgnn.pt already exists AND is marked
# completed=True (e.g. restored from Drive by restore_from_drive_if_present()
# in Section 2), fit() loads it and skips training entirely; otherwise it
# trains normally, saving the best-so-far state dict to that path every time
# validation loss improves. sync_to_drive() below is still needed afterward
# -- the trainer only writes locally, it has no notion of Google Drive.
vmd_mc = vmd_mfgnn_config['model']
vmd_mfgnn_checkpoint = Path('results/checkpoints/vmd_mfgnn.pt')

set_seed(seed)
model = VMDMFGNN(
    num_vars=num_vars, num_modes=num_modes,
    hidden_dim=vmd_mc['hidden_dim'], num_heads=vmd_mc['num_heads'],
    num_gnn_layers=vmd_mc['num_gnn_layers'],
    temporal_layers=vmd_mc['temporal_layers'],
    dropout=vmd_mc['dropout'], horizons=horizons,
    graph_type=vmd_mc['graph_type'],
)
trainer = VMDMFGNNTrainer(model, vmd_mfgnn_config)

print('Training VMD-MFGNN (proposed model)...')
t0 = time.time()
history = trainer.fit(data['train_loader'], data['val_loader'],
                       checkpoint_path=vmd_mfgnn_checkpoint)
if history.get('resumed_from_checkpoint'):
    print(f'[resume] loaded checkpoint from {vmd_mfgnn_checkpoint} -- training skipped')
else:
    print(f'VMD-MFGNN training done in {time.time() - t0:.1f}s')

test_results = trainer.evaluate(data['test_loader'])
test_results['name'] = 'VMD-MFGNN'
all_results['VMD-MFGNN'] = test_results
print('VMD-MFGNN results:', test_results)

# predict() also (re)populates the model's internal attention/graph state
# needed for get_attention_weights()/get_learned_graphs() below, so always
# run it even when weights were loaded from a checkpoint.
vmd_preds = trainer.predict(data['test_loader'])
test_ys = []
for _, y in data['test_loader']:
    test_ys.append(y.numpy() if isinstance(y, torch.Tensor) else y)
test_true = np.concatenate(test_ys, axis=0)
for i, h in enumerate(horizons):
    np.save(pred_dir / f'VMD-MFGNN_{h}.npy', vmd_preds[str(h)])
    np.save(pred_dir / f'VMD-MFGNN_{h}_true.npy', test_true[:, i])

learned_graphs = model.get_learned_graphs()
attention_weights = model.get_attention_weights()
if learned_graphs:
    torch.save(learned_graphs, interp_dir / 'learned_graphs.pt')
if attention_weights is not None:
    torch.save(attention_weights, interp_dir / 'attention_weights.pt')

sync_to_drive('results', note='after VMD-MFGNN')

In [ ]:
# ---- Raw-price baselines (ARIMA, LSTM, Transformer, SimpleMTGNN, MTGNN,
# XGBoost), each independently resumable via saved-prediction files ----
raw_baselines = [
    ('ARIMA', lambda: ARIMABaseline(base_cfg)),
    ('LSTM', lambda: LSTMBaseline(base_cfg)),
    ('Transformer', lambda: TransformerBaseline(base_cfg)),
    ('SimpleMTGNN', lambda: SimpleMTGNN(base_cfg)),
    # Faithful MTGNN reproduction (see src/models/baselines.py's MTGNNBaseline
    # docstring), kept alongside SimpleMTGNN as a second, higher-fidelity row
    # rather than a replacement.
    ('MTGNN', lambda: MTGNNBaseline(base_cfg)),
    ('XGBoost', lambda: XGBoostBaseline(base_cfg)),
]

for name, make_model in raw_baselines:
    if _predictions_exist(name, horizons):
        print(f'[resume] {name}: predictions already on disk -- skipping training, '
              f'recomputing metrics from saved arrays')
        res = _metrics_from_saved_predictions(name, horizons)
    else:
        set_seed(seed)  # per-model seeding: each model's init+shuffle RNG stream
                         # is independent of whichever model ran before it.
        bl = make_model()
        res = run_baseline(bl, data['raw_train_loader'], data['raw_val_loader'],
                            data['raw_test_loader'], name)
        sync_to_drive('results', note=f'after baseline {name}')
    all_results[name] = res
    print(name, '->', {k: v for k, v in res.items() if k != 'name'})

In [ ]:
# ---- VMD baseline (VMD-LSTM: needs VMD-mode loaders, not raw) ----
name = 'VMD-LSTM'
if _predictions_exist(name, horizons):
    print(f'[resume] {name}: predictions already on disk -- skipping training, '
          f'recomputing metrics from saved arrays')
    res = _metrics_from_saved_predictions(name, horizons)
else:
    set_seed(seed)
    bl = VMDLSTMBaseline(base_cfg)
    res = run_baseline(bl, data['train_loader'], data['val_loader'],
                        data['test_loader'], name)
    sync_to_drive('results', note=f'after baseline {name}')
all_results[name] = res
print(name, '->', {k: v for k, v in res.items() if k != 'name'})

In [ ]:
# ---- Save aggregated results + Diebold-Mariano significance table ----
save_results(all_results, 'results/all_results.json')
print('All results saved to results/all_results.json')

print_results_table(all_results, horizons)

baseline_names = [name for name, _ in raw_baselines] + ['VMD-LSTM']
significance_table = run_significance_tests('VMD-MFGNN', baseline_names, horizons)

sync_to_drive('results', note='after main-results save + significance table')

# Verify the significance table was actually produced (don't assume -- confirm).
sig_path = Path('results/significance_table.json')
assert sig_path.exists(), 'results/significance_table.json was not created!'
with open(sig_path) as f:
    sig_table = json.load(f)
print(f'significance_table.json has {len(sig_table)} entries')
nan_entries = [
    k for k, v in sig_table.items()
    if v.get('dm_stat') is None or v.get('p_value') is None
    or (isinstance(v.get('dm_stat'), float) and v['dm_stat'] != v['dm_stat'])
    or (isinstance(v.get('p_value'), float) and v['p_value'] != v['p_value'])
]
if nan_entries:
    print(f'*** WARNING: {len(nan_entries)}/{len(sig_table)} entries have NaN '
          f'dm_stat/p_value: {nan_entries} -- sanity-check before reporting. ***')
else:
    print('No NaN entries in significance_table.json.')

## Section 6: Ablation Studies (8 variants)

Replicates `src/trainer.py`'s `run_ablation_studies()` per-variant
construction exactly, as individually-resumable, individually-Drive-synced
cells (`run_ablation_studies()` itself is one call with no way to sync
between variants -- not used directly here for that reason). Every variant's
completion reaches Drive before the next variant starts.

Variants: `full_model`, `no_vmd_raw_price_matched_dim`,
`no_vmd_raw_price_matched_params`, `pooled_graph_matched_dim`,
`pooled_graph_matched_params`, `correlation_graph`, `full_model_graphfix`,
`full_model_graphfix_temp`. Uses the **untuned** `config` (`mc = config['model']`)
throughout -- ablation variants are never HPO-tuned, matching
`run_ablation_studies()`'s own `mc = config["model"]  # raw config, NOT
vmd_mfgnn_config` comment.

The last two variants (`full_model_graphfix` / `full_model_graphfix_temp`)
require temporarily setting `config['training']['no_decay_graph_embeddings']
= True` before constructing the model and trainer (the trainer reads this
flag at construction time, not at `.fit()` time), then restoring it
immediately after -- variants run sequentially, never concurrently, so this
mutate/restore is safe.


In [ ]:
mc = config['model']  # raw config, NOT vmd_mfgnn_config -- ablations are never HPO-tuned
ablation_seed = config['training']['seed']
ablation_results = {}
ablation_checkpoint_dir = Path('results/checkpoints')
ablation_checkpoint_dir.mkdir(parents=True, exist_ok=True)
ablation_pred_dir = Path('results/ablation_predictions')
ablation_pred_dir.mkdir(parents=True, exist_ok=True)


def make_vmd_mfgnn(num_modes_, graph_type_, hidden_dim_=None,
                    normalize_graph_embeddings_=False,
                    use_graph_temperature_=False):
    return VMDMFGNN(
        num_vars=num_vars, num_modes=num_modes_,
        hidden_dim=hidden_dim_ if hidden_dim_ is not None else mc['hidden_dim'],
        num_heads=mc['num_heads'],
        num_gnn_layers=mc['num_gnn_layers'],
        temporal_layers=mc['temporal_layers'],
        dropout=mc['dropout'], horizons=horizons,
        graph_type=graph_type_,
        normalize_graph_embeddings=normalize_graph_embeddings_,
        use_graph_temperature=use_graph_temperature_,
    )


def make_pooled(hidden_dim_=None):
    return PooledGraphMFGNN(
        num_vars=num_vars, num_modes=num_modes,
        hidden_dim=hidden_dim_ if hidden_dim_ is not None else mc['hidden_dim'],
        num_heads=mc['num_heads'],
        num_gnn_layers=mc['num_gnn_layers'],
        temporal_layers=mc['temporal_layers'],
        dropout=mc['dropout'], horizons=horizons,
        graph_type='learned',
    )


def run_variant(key, model_, train_loader, val_loader, test_loader):
    # Checkpointing is trainer-native: fit(checkpoint_path=...) loads and
    # skips training if results/checkpoints/{key}.pt already exists AND is
    # marked completed=True (e.g. restored from Drive), otherwise trains and
    # checkpoints the best-so-far state dict there as it goes.
    variant_trainer = VMDMFGNNTrainer(model_, config)
    variant_trainer.fit(train_loader, val_loader,
                         checkpoint_path=ablation_checkpoint_dir / f'{key}.pt')
    results = variant_trainer.evaluate(test_loader)
    results['name'] = key
    results['final_epoch'] = variant_trainer.final_epoch
    ablation_results[key] = results

    # Archive per-sample predictions/targets so a paired significance test
    # can be computed for ablation comparisons after the fact.
    preds = variant_trainer.predict(test_loader)
    ys = []
    for _, y in test_loader:
        ys.append(y.numpy() if isinstance(y, torch.Tensor) else y.cpu().numpy())
    true = np.concatenate(ys, axis=0)
    for i, h in enumerate(model_.horizons):
        pred_h = preds[str(h)]
        true_h = true[:, i]
        np.save(ablation_pred_dir / f'{key}_{h}.npy', pred_h)
        np.save(ablation_pred_dir / f'{key}_{h}_true.npy', true_h)

    save_results(ablation_results, 'results/ablation_results.json')
    sync_to_drive('results', note=f'after ablation variant {key}')
    print(key, '->', results)
    return variant_trainer

In [ ]:
# (a) full_model: standard VMD-MFGNN, learned per-band graphs, real VMD data.
print('Ablation: full_model')
set_seed(ablation_seed)
full_model = make_vmd_mfgnn(num_modes, 'learned')
full_params = count_parameters(full_model)
print(f"full_model param count: {full_params:,} (hidden_dim={mc['hidden_dim']})")
run_variant('full_model', full_model, data['train_loader'],
            data['val_loader'], data['test_loader'])

In [ ]:
# (b1) no_vmd_raw_price_matched_dim: genuine no-VMD baseline at the config's
# shared hidden_dim (capacity-UNMATCHED vs. full_model, kept for continuity
# with the pre-fix ablation). Trained/evaluated on RawPriceDataset-derived
# loaders -- data that has never touched VMD decomposition -- via a
# num_modes=1 VMDMFGNN wrapped to unsqueeze a singleton mode dim onto the 3D
# raw batches.
print('Ablation: no_vmd_raw_price_matched_dim')
set_seed(ablation_seed)
no_vmd_dim_inner = make_vmd_mfgnn(1, 'learned')
no_vmd_dim_params = count_parameters(no_vmd_dim_inner)
print(f"no_vmd_raw_price_matched_dim param count: {no_vmd_dim_params:,} "
      f"vs full_model's {full_params:,} "
      f"({(no_vmd_dim_params - full_params) / full_params * 100:+.1f}%)")
no_vmd_dim_model = _UnsqueezeModeWrapper(no_vmd_dim_inner)
run_variant('no_vmd_raw_price_matched_dim', no_vmd_dim_model,
            data['raw_train_loader'], data['raw_val_loader'], data['raw_test_loader'])

In [ ]:
# (b2) no_vmd_raw_price_matched_params: same no-VMD setup, but at a
# hidden_dim searched (via _find_matched_hidden_dim) to bring parameter
# count close to full_model's (capacity-MATCHED).
def _build_no_vmd(h):
    return make_vmd_mfgnn(1, 'learned', hidden_dim_=h)

_matched_h, _matched_c = _find_matched_hidden_dim(_build_no_vmd, full_params, mc['num_heads'])
print(f"matched no_vmd_raw_price to hidden_dim={_matched_h}, {_matched_c:,} params "
      f"vs full_model's {full_params:,} ({(_matched_c - full_params) / full_params * 100:+.1f}%)")
print('Ablation: no_vmd_raw_price_matched_params')
set_seed(ablation_seed)
no_vmd_params_inner = _build_no_vmd(_matched_h)
no_vmd_params_model = _UnsqueezeModeWrapper(no_vmd_params_inner)
run_variant('no_vmd_raw_price_matched_params', no_vmd_params_model,
            data['raw_train_loader'], data['raw_val_loader'], data['raw_test_loader'])

In [ ]:
# (c1) pooled_graph_matched_dim: PooledGraphMFGNN -- ONE pooled graph instead
# of K per-band graphs, at the config's shared hidden_dim (capacity-UNMATCHED
# vs. full_model, kept for continuity). Trained/evaluated on the same VMD-mode
# loaders as full_model.
print('Ablation: pooled_graph_matched_dim')
set_seed(ablation_seed)
pooled_dim_model = make_pooled()
pooled_dim_params = count_parameters(pooled_dim_model)
print(f"pooled_graph_matched_dim param count: {pooled_dim_params:,} "
      f"vs full_model's {full_params:,} "
      f"({(pooled_dim_params - full_params) / full_params * 100:+.1f}%)")
run_variant('pooled_graph_matched_dim', pooled_dim_model,
            data['train_loader'], data['val_loader'], data['test_loader'])

In [ ]:
# (c2) pooled_graph_matched_params: same pooled-graph setup, but at a
# hidden_dim searched to bring parameter count close to full_model's
# (capacity-MATCHED) -- the honest version of the paper's central
# "per-band beats pooled" comparison.
_matched_h, _matched_c = _find_matched_hidden_dim(make_pooled, full_params, mc['num_heads'])
print(f"matched pooled_graph to hidden_dim={_matched_h}, {_matched_c:,} params "
      f"vs full_model's {full_params:,} ({(_matched_c - full_params) / full_params * 100:+.1f}%)")
print('Ablation: pooled_graph_matched_params')
set_seed(ablation_seed)
pooled_params_model = make_pooled(_matched_h)
run_variant('pooled_graph_matched_params', pooled_params_model,
            data['train_loader'], data['val_loader'], data['test_loader'])

In [ ]:
# (d) correlation_graph: VMDMFGNN with graph_type='correlation', per-band
# frozen (train-split-only, computed once) correlation adjacency threaded
# through via the trainer's _forward/_freeze_correlation_adjs helpers. Same
# VMD-mode loaders.
print('Ablation: correlation_graph')
set_seed(ablation_seed)
corr_model = make_vmd_mfgnn(num_modes, 'correlation')
run_variant('correlation_graph', corr_model,
            data['train_loader'], data['val_loader'], data['test_loader'])

In [ ]:
# (e) full_model_graphfix: the anti-collapse fix on its own -- unit-normalized
# graph embeddings, excluded from weight decay. no_decay_graph_embeddings
# must be set on `config` BEFORE VMDMFGNNTrainer(...) is constructed (the
# trainer's optimizer param-group setup reads it at construction time, not
# at .fit() time), so it is set immediately before run_variant and restored
# in a finally block -- variants run sequentially, so there's no concurrency
# hazard, but an unguarded restore is still an EXCEPTION hazard: if
# run_variant raises partway through (OOM, a NaN loss, a Drive-sync error --
# all real risks over an unattended multi-hour run) and the notebook is
# later resumed in the SAME kernel rather than a full restart, the flag
# would otherwise be left stuck at True and silently affect every later
# cell that reuses this same `config` object.
print('Ablation: full_model_graphfix')
set_seed(ablation_seed)
_no_decay_saved = config['training'].get('no_decay_graph_embeddings', False)
config['training']['no_decay_graph_embeddings'] = True
try:
    graphfix_model = make_vmd_mfgnn(num_modes, 'learned', normalize_graph_embeddings_=True)
    run_variant('full_model_graphfix', graphfix_model,
                data['train_loader'], data['val_loader'], data['test_loader'])
finally:
    config['training']['no_decay_graph_embeddings'] = _no_decay_saved


In [ ]:
# (f) full_model_graphfix_temp: (e) plus a learnable per-band softmax
# temperature, tau=exp(log_temperature) -- the "best available" graph fix,
# built to survive weight decay where a raw nn.Parameter temperature would
# not. log_temperature is caught by the same no_decay_graph_embeddings
# name-matcher as emb1/emb2, so it reuses the same config flag as (e). Same
# try/finally reasoning as (e) above.
print('Ablation: full_model_graphfix_temp')
set_seed(ablation_seed)
config['training']['no_decay_graph_embeddings'] = True
try:
    graphfix_temp_model = make_vmd_mfgnn(num_modes, 'learned',
                                          normalize_graph_embeddings_=True,
                                          use_graph_temperature_=True)
    run_variant('full_model_graphfix_temp', graphfix_temp_model,
                data['train_loader'], data['val_loader'], data['test_loader'])
finally:
    config['training']['no_decay_graph_embeddings'] = _no_decay_saved

print('All 8 ablation variants complete:', sorted(ablation_results.keys()))


## Section 7: Null-Control Experiment

**Why this experiment exists** (see `src/experiments.py`'s
`run_null_control_experiment` docstring): is the per-band learned-graph
collapse-to-uniform a property of THIS decomposition's structure, or would
it happen for ANY input fed to this mechanism? Three arms, same
architecture/hyperparameters/seed/epoch budget:
- `real` -- the actual VMD-decomposed input (the control arm; a null result
  is uninterpretable without this).
- `shuffle` -- every non-target channel's sample axis independently permuted,
  destroying cross-variable co-movement while leaving each channel's own
  temporal dynamics intact.
- `gaussian` -- every non-target channel replaced with matched-mean/std
  Gaussian noise.

The target channel (copper) is always preserved in every arm, so the model
still trains on genuine signal through the temporal path -- only the graph's
neighbours carry no real information in the null arms. If `shuffle`/`gaussian`
collapse identically to `real`, the collapse is downstream of the
decomposition choice entirely, not a finding about VMD bands specifically.

Already resumable per-cell internally (`run_resumable_grid`) -- `on_cell_done`
below is this notebook's Drive-sync callback, `make_cell_sync`, from Section 2.
`hpo_best_params_path` is overridden to `results/hpo_best_params.json` --
this run's OWN HPO output (the function's own default points at a stale
pre-this-session N=8 result and must not be used).


In [ ]:
null_control_results_dir = 'results/robustness/null_control'
null_control_results = exp.run_null_control_experiment(
    config,
    modes=('real', 'shuffle', 'gaussian'),
    epochs=40,
    hpo_best_params_path='results/hpo_best_params.json',
    results_dir=null_control_results_dir,
    data=data,
    on_cell_done=make_cell_sync(null_control_results_dir, 'null_control'),
)
print('Null-control experiment complete:')
for row in null_control_results:
    tm = row.get('test_metrics', {})
    print(' ', row.get('mode'), '-> avg_mse:', tm.get('avg_mse'))

## Section 8: Figure Generation

Pulls the trained VMD-MFGNN model's attention weights (converting to numpy
if still a torch tensor) and calls `generate_all_figures` to produce every
paper figure from the results computed above.


In [ ]:
attn_weights = trainer.model.get_attention_weights()
if hasattr(attn_weights, 'cpu'):
    attn_weights = attn_weights.cpu().numpy()
elif attn_weights is not None and not isinstance(attn_weights, np.ndarray):
    attn_weights = np.array(attn_weights)

fig_dir = Path('results/figures')
fig_dir.mkdir(parents=True, exist_ok=True)

generate_all_figures(
    config, data, all_results, ablation_results,
    output_dir=str(fig_dir),
    model=trainer.model,
    attn_weights=attn_weights,
)

print('Figures written to', fig_dir)
print(sorted(os.listdir(fig_dir)))

sync_to_drive('results', note='after figure generation')

## Section 9: Final Sync + Zip + Download

Full final mirror of `data/` and `results/` to Drive (idempotent -- safe to
re-run), plus a zip of `results/` for a one-click local download.


In [ ]:
sync_to_drive('results', note='FINAL sync')
sync_data_cache_to_drive(note='FINAL sync')
print('Final sync complete. Drive folder:', DRIVE_ROOT)
print(sorted(os.listdir(DRIVE_ROOT)))

import shutil as _shutil
_shutil.make_archive('/content/copper_paper1_results', 'zip', 'results')

from google.colab import files
files.download('/content/copper_paper1_results.zip')

## Done -- Checklist

Check, in order, once the run has finished:

1. `results/hpo_best_params.json` + `results/hpo_trials.csv` -- HPO winner
   and full trial history.
2. `results/checkpoints/vmd_mfgnn.pt` -- `completed: True`.
3. `results/all_results.json` -- VMD-MFGNN + 8 baselines x 4 horizons.
4. `results/significance_table.json` -- Diebold-Mariano vs. each baseline;
   re-check for NaN entries (Section 5's own check should have already
   flagged any).
5. `results/predictions/*.npy` -- per-model, per-horizon prediction + ground
   truth arrays (the source of truth for re-verifying any reported metric).
6. `results/interpretability/learned_graphs.pt`,
   `results/interpretability/attention_weights.pt`.
7. `results/ablation_results.json` + `results/checkpoints/{variant}.pt` for
   all 8 variants (`full_model`, `no_vmd_raw_price_matched_dim`,
   `no_vmd_raw_price_matched_params`, `pooled_graph_matched_dim`,
   `pooled_graph_matched_params`, `correlation_graph`, `full_model_graphfix`,
   `full_model_graphfix_temp`).
8. `results/ablation_predictions/*.npy` -- per-variant, per-horizon
   prediction + ground truth arrays.
9. `results/robustness/null_control/null_control_results.jsonl` -- 3 rows
   (`real`, `shuffle`, `gaussian`), each with `test_metrics` and a
   `diagnostic` block.
10. `results/figures/` -- all 5 paper figures (`.pdf` + `.png`).
11. `/content/copper_paper1_results.zip` downloaded locally (or restore from
    `MyDrive/Copper_Paper1.2/` directly).
12. Every checkpoint file referenced above should show `completed: True`
    when loaded (`torch.load(path)['completed']`) -- a `False` means that
    unit of work was interrupted mid-training and should be re-run before
    trusting its numbers.
